In [10]:

import pandas as pd
import numpy as np
import json
import re
import html

from pyproj import Transformer

# ============================================================
# OPERATOR ALIASES
# ============================================================

OPERATOR_ALIASES = {
    "TELEFONICA MOVILES ESPAÑA, S.A.U.": "movistar",
    "TELEFONICA MOVILES ESPAÑA, S.A.": "movistar",
    "ORANGE ESPAGNE, S.A.": "orange",
    "ORANGE ESPAGNE, S.A.U.": "orange",
    "VODAFONE ESPAÑA, S.A.U.": "vodafone",
    "VODAFONE ESPAÑA, S.A.": "vodafone",
    "DIGI SPAIN TELECOM, S.L.": "digi",
    "DS MOBILE NETWORKS, S.L.": "digimobil"
}


def get_operator_alias(operator):

    normalized = operator.strip().upper()

    alias = OPERATOR_ALIASES.get(normalized)

    if alias is None:
        # Fallback for operators not included in the dictionary
        alias = re.sub(r"[^a-z0-9]+", "_",
                       operator.lower()).strip("_")

    return alias


# ============================================================
# 1. LOAD AND FILTER REAL RRHs
# ============================================================

def load_rrhs(csv_path, selected_operator):

    df = pd.read_csv(csv_path, dtype=str)

    # Normalize column names
    df.columns = [
        re.sub(r"\s+", "_",
               html.unescape(str(c)).replace("*", "").strip().lower())
        for c in df.columns
    ]

    rrhs = []

    for _, row in df.iterrows():

        # Find the selected operator and its corresponding technology
        for i in range(1, 5):

            operator = row.get(f"operator_{i}", "")

            if pd.isna(operator):
                continue

            if str(operator).strip().casefold() != selected_operator.strip().casefold():
                continue

            technology = str(row.get(f"technology_op{i}", "")).strip().upper()

            if "4G" not in technology and "5G" not in technology:
                continue

            # Parse coordinates: latitude, longitude
            coordinates = str(row["coordinates"]).split(",")

            lat = float(coordinates[0].strip())
            lon = float(coordinates[1].strip())

            # Parse Site ID list
            try:
                site_ids = json.loads(row["site_id"])
                site_id = site_ids[0] if site_ids else "UNKNOWN"
            except (ValueError, TypeError):
                site_id = str(row["site_id"])

            rrhs.append({
                "rrh_id": f"RRH_{len(rrhs)+1:03d}",
                "site_id": site_id,
                "operator": operator,
                "technology": technology,
                "latitude": lat,
                "longitude": lon
            })

            # One RRH per selected operator and physical site
            break

    result = pd.DataFrame(rrhs)

    if result.empty:
        raise ValueError(
            f"No 4G/5G stations found for {selected_operator}"
        )

    # Remove duplicate geographic positions
    result = result.drop_duplicates(
        subset=["latitude", "longitude"]
    ).reset_index(drop=True)

    # Regenerate RRH IDs after filtering
    result["rrh_id"] = [
        f"RRH_{i+1:03d}" for i in range(len(result))
    ]

    return result


# ============================================================
# 2. CONVERT GEOGRAPHIC COORDINATES TO METERS
# ============================================================

def project_coordinates(rrhs):

    # EPSG:25831 = UTM 31N (suitable for Barcelona)
    transformer = Transformer.from_crs(
        "EPSG:4326",
        "EPSG:25831",
        always_xy=True
    )

    x, y = transformer.transform(
        rrhs["longitude"].values,
        rrhs["latitude"].values
    )

    return np.column_stack((x, y))


# ============================================================
# 3. FARTHEST-FIRST K-CENTER
# ============================================================

def farthest_first_k_center(X, K):

    n = len(X)

    # Candidate positions: unique RRH locations
    _, candidate_idx = np.unique(
        X, axis=0, return_index=True
    )

    candidates = X[candidate_idx]
    n_candidates = len(candidates)

    if K < 1 or K > n_candidates:
        raise ValueError(
            f"K must be between 1 and {n_candidates}"
        )

    # Initial center: point farthest from the centroid
    centroid = candidates.mean(axis=0)

    first = np.argmax(
        np.linalg.norm(candidates - centroid, axis=1)
    )

    selected = [first]

    # Distance from every candidate to its nearest center
    nearest_distances = np.linalg.norm(
        candidates - candidates[first], axis=1
    )

    # Iteratively select the farthest candidate
    for _ in range(1, K):

        next_center = np.argmax(nearest_distances)

        selected.append(next_center)

        new_distances = np.linalg.norm(
            candidates - candidates[next_center], axis=1
        )

        nearest_distances = np.minimum(
            nearest_distances, new_distances
        )

    # Return original RRH indices
    return candidate_idx[selected]


# ============================================================
# 4. ASSIGN EACH RRH TO ITS NEAREST BBU POOL
# ============================================================

def assign_rrhs_to_pools(rrhs, X, selected_indices):

    centers = X[selected_indices]

    # Distance matrix: RRHs x BBU pools
    distance_matrix = np.linalg.norm(
        X[:, np.newaxis, :] - centers[np.newaxis, :, :],
        axis=2
    )

    # Nearest BBU pool for every RRH
    assigned_pool = np.argmin(distance_matrix, axis=1)

    # Distance to the assigned BBU pool
    assigned_distance = distance_matrix[
        np.arange(len(X)), assigned_pool
    ]

    result = rrhs.copy()

    result["bbu_pool"] = [
        f"BBU_{i+1:02d}" for i in assigned_pool
    ]

    result["distance_to_bbu_km"] = assigned_distance / 1000

    # Build BBU pool information
    pools = rrhs.iloc[selected_indices].copy().reset_index(drop=True)

    pools["bbu_pool"] = [
        f"BBU_{i+1:02d}" for i in range(len(pools))
    ]

    pools["num_assigned_rrhs"] = [
        np.sum(assigned_pool == i)
        for i in range(len(selected_indices))
    ]

    return result, pools


# ============================================================
# 5. MAIN PROGRAM
# ============================================================

if __name__ == "__main__":

    OUTPUT_PATH = "../data/processed/"
    CSV_PATH = OUTPUT_PATH + "350_CELLS.csv"

    OPERATOR = "TELEFONICA MOVILES ESPAÑA, S.A.U."

    K = 3  # Desired number of BBU pools

    # --------------------------------------------------------
    # OPERATOR ALIAS
    # --------------------------------------------------------

    operator_alias = get_operator_alias(OPERATOR)

    # --------------------------------------------------------
    # LOAD REAL RRHs
    # --------------------------------------------------------

    rrhs = load_rrhs(CSV_PATH, OPERATOR)

    print(f"\nSelected operator: {OPERATOR}")
    print(f"Operator alias: {operator_alias}")
    print(f"Number of real RRHs: {len(rrhs)}")

    # --------------------------------------------------------
    # PROJECT COORDINATES
    # --------------------------------------------------------

    X = project_coordinates(rrhs)

    # --------------------------------------------------------
    # SELECT BBU POOL LOCATIONS
    # --------------------------------------------------------

    selected_indices = farthest_first_k_center(X, K)

    # --------------------------------------------------------
    # ASSIGN RRHs TO BBU POOLS
    # --------------------------------------------------------

    assignments, pools = assign_rrhs_to_pools(
        rrhs, X, selected_indices
    )

    # --------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------

    print("\nBBU POOL LOCATIONS:")

    print(pools[
        ["bbu_pool", "site_id", "latitude",
         "longitude", "num_assigned_rrhs"]
    ].to_string(index=False))

    print("\nRRH ASSIGNMENTS:")

    print(assignments[
        ["rrh_id", "site_id", "bbu_pool",
         "distance_to_bbu_km"]
    ].to_string(index=False))

    # --------------------------------------------------------
    # PERFORMANCE INDICATORS
    # --------------------------------------------------------

    print("\nPERFORMANCE:")

    print(f"Maximum distance: "
          f"{assignments['distance_to_bbu_km'].max():.3f} km")

    print(f"Average distance: "
          f"{assignments['distance_to_bbu_km'].mean():.3f} km")

    # --------------------------------------------------------
    # EXPORT RESULTS WITH OPERATOR AND K
    # --------------------------------------------------------

    assignments_file = ( OUTPUT_PATH +
        f"rrh_bbu_k{K}_assignments_{operator_alias}.csv"
    )

    pools_file = (
        OUTPUT_PATH +
        f"bbu_pool_k{K}_locations_{operator_alias}.csv"
    )

    assignments.to_csv(assignments_file, index=False)

    pools.to_csv(pools_file, index=False)

    print("\nRESULTS EXPORTED SUCCESSFULLY:")
    print(f" - {assignments_file}")
    print(f" - {pools_file}")






Selected operator: TELEFONICA MOVILES ESPAÑA, S.A.U.
Operator alias: movistar
Number of real RRHs: 194

BBU POOL LOCATIONS:
bbu_pool site_id  latitude  longitude  num_assigned_rrhs
  BBU_01 0800359 41.442799   1.844017                 37
  BBU_02 0805243 41.411392   2.226772                127
  BBU_03 0802498 41.265598   1.994106                 30

RRH ASSIGNMENTS:
 rrh_id site_id bbu_pool  distance_to_bbu_km
RRH_001 0802073   BBU_02           10.143098
RRH_002 0801936   BBU_03            7.931954
RRH_003 0802421   BBU_02            6.810097
RRH_004 0807064   BBU_02            5.332829
RRH_005  075726   BBU_03           11.625558
RRH_006 0803052   BBU_02            7.626745
RRH_007 0808372   BBU_01           19.400649
RRH_008  075723   BBU_01           12.772734
RRH_009  070903   BBU_02            1.746518
RRH_010 0806665   BBU_03            5.109368
RRH_011 0800273   BBU_02            9.580731
RRH_012 0808423   BBU_02            7.231443
RRH_013 0808453   BBU_02            3.727686

In [11]:

import pandas as pd
import folium
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from IPython.display import display, IFrame

# ============================================================
# 1. LOAD CSV FILES
# ============================================================
BASE_PATH = "../data/processed/"
basename = "rrh_bbu_k3_assignments_movistar.csv"
rrh_file = BASE_PATH + basename
bbu_file = BASE_PATH + "bbu_pool_k3_locations_movistar.csv"

rrhs = pd.read_csv(rrh_file)
bbus = pd.read_csv(bbu_file)

print("RRH columns:", rrhs.columns.tolist())
print("BBU columns:", bbus.columns.tolist())

print(f"\nNumber of RRHs: {len(rrhs)}")
print(f"Number of BBU pools: {len(bbus)}")


# ============================================================
# 2. VALIDATE REQUIRED COLUMNS
# ============================================================

required_rrh = ["latitude", "longitude", "bbu_pool"]
required_bbu = ["latitude", "longitude"]

for col in required_rrh:
    if col not in rrhs.columns:
        raise ValueError(f"Missing column in RRH CSV: {col}")

for col in required_bbu:
    if col not in bbus.columns:
        raise ValueError(f"Missing column in BBU CSV: {col}")


# ============================================================
# 3. GENERATE COLORS FOR EACH BBU POOL
# ============================================================

pool_ids = sorted(
    rrhs["bbu_pool"].dropna().unique(),
    key=lambda x: str(x)
)

cmap = plt.get_cmap("tab10", max(len(pool_ids), 1))

pool_colors = {
    pool_id: to_hex(cmap(i))
    for i, pool_id in enumerate(pool_ids)
}


# ============================================================
# 4. MAP CENTER
# ============================================================

center_lat = rrhs["latitude"].mean()
center_lon = rrhs["longitude"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    #tiles="OpenStreetMap"
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri Street Map",
    overlay=False,
    control=True
).add_to(m)


# ============================================================
# 5. RRH MARKERS, COLORED BY ASSIGNED BBU POOL
# ============================================================

rrh_layer = folium.FeatureGroup(
    name="RRHs (colored by assigned BBU pool)",
    show=True
)

for _, row in rrhs.iterrows():

    pool_id = row["bbu_pool"]
    color = pool_colors.get(pool_id, "gray")

    popup_text = f"""
    <b>RRH:</b> {row.get('rrh_id', 'N/A')}<br>
    <b>Site ID:</b> {row.get('site_id', 'N/A')}<br>
    <b>Assigned BBU:</b> {pool_id}<br>
    <b>Distance:</b> {row.get('distance_to_bbu_km', 'N/A')} km
    """

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        weight=1,
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=f"RRH - BBU Pool {pool_id}"
    ).add_to(rrh_layer)

rrh_layer.add_to(m)


# ============================================================
# 6. BBU POOL MARKERS
# ============================================================

bbu_layer = folium.FeatureGroup(
    name="BBU Pools",
    show=True
)

for i, row in bbus.iterrows():

    pool_id = row.get("bbu_pool", i + 1)

    popup_text = f"""
    <b>BBU Pool:</b> {pool_id}<br>
    <b>Site ID:</b> {row.get('site_id', 'N/A')}<br>
    <b>Assigned RRHs:</b> {row.get('num_assigned_rrhs', 'N/A')}
    """

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=f"BBU Pool {pool_id}",
        icon=folium.Icon(
            color="black",
            icon="server",
            prefix="fa"
        )
    ).add_to(bbu_layer)

bbu_layer.add_to(m)


# ============================================================
# 7. RRH - BBU CONNECTION LINES
# ============================================================

connection_layer = folium.FeatureGroup(
    name="RRH - BBU connections",
    show=True
)

# Build a dictionary with the coordinates of each BBU pool
bbu_coordinates = {}

for i, row in bbus.iterrows():
    pool_id = row.get("bbu_pool", i + 1)

    bbu_coordinates[pool_id] = [
        row["latitude"],
        row["longitude"]
    ]

for _, row in rrhs.iterrows():

    pool_id = row["bbu_pool"]

    if pool_id not in bbu_coordinates:
        continue

    folium.PolyLine(
        locations=[
            [row["latitude"], row["longitude"]],
            bbu_coordinates[pool_id]
        ],
        color=pool_colors.get(pool_id, "gray"),
        weight=1.5,
        opacity=0.5
    ).add_to(connection_layer)

connection_layer.add_to(m)


# ============================================================
# 8. MAP CONTROLS
# ============================================================

folium.LayerControl(collapsed=False).add_to(m)

# Fit map to all RRHs
m.fit_bounds([
    [rrhs["latitude"].min(), rrhs["longitude"].min()],
    [rrhs["latitude"].max(), rrhs["longitude"].max()]
])


filename = "../output/" + basename.replace(".csv", "_map.html")
m.save(filename)
#display(IFrame(filename, width="100%", height="600px"))
display(m)


# ============================================================
# 9. ASSIGNMENT SUMMARY
# ============================================================

print("\nRRH assignment summary:")

summary = (
    rrhs.groupby("bbu_pool")
    .agg(
        assigned_RRHs=("bbu_pool", "count"),
        mean_distance_km=("distance_to_bbu_km", "mean"),
        max_distance_km=("distance_to_bbu_km", "max")
    )
    .round(3)
)

display(summary)


RRH columns: ['rrh_id', 'site_id', 'operator', 'technology', 'latitude', 'longitude', 'bbu_pool', 'distance_to_bbu_km']
BBU columns: ['rrh_id', 'site_id', 'operator', 'technology', 'latitude', 'longitude', 'bbu_pool', 'num_assigned_rrhs']

Number of RRHs: 194
Number of BBU pools: 3



RRH assignment summary:


,assigned_RRHs,mean_distance_km,max_distance_km
bbu_pool,,,
BBU_01,37,13.780,22.771
BBU_02,127,7.870,18.829
BBU_03,30,8.624,14.074
